# 00 — Catalog & Schema Setup

One-time bootstrap. Run under an identity with CREATE CATALOG on the metastore.
Idempotent — safe to re-run. `01_bronze` and `02_silver` assume these exist.

In [0]:
dbutils.widgets.text("catalog", "robocall", "Catalog name")
dbutils.widgets.text("managed_location", "", "Managed location (optional)")

CATALOG = dbutils.widgets.get("catalog").strip()
MANAGED_LOCATION = dbutils.widgets.get("managed_location").strip()
SCHEMAS = ["source_data", "bronze", "silver", "gold"]

if not CATALOG:
    raise ValueError("catalog widget must not be empty")

print(f"Catalog:  {CATALOG}")
print(f"Location: {MANAGED_LOCATION or '(metastore default)'}")

In [0]:
ddl = f"CREATE CATALOG IF NOT EXISTS {CATALOG}"
if MANAGED_LOCATION:
    ddl += f" MANAGED LOCATION '{MANAGED_LOCATION}'"
ddl += " COMMENT 'FCC robocall complaint pipeline'"

spark.sql(ddl)
print(f"catalog ready: {CATALOG}")

In [0]:
COMMENTS = {
    "source_data": "Raw landing zone for FCC complaint files",
    "bronze": "Ingested as-is, all columns string, no casting",
    "silver": "Typed, deduplicated, cleaned complaint tickets",
    "gold": "Aggregates and serving tables",
}

for schema in SCHEMAS:
    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema} "
        f"COMMENT '{COMMENTS[schema]}'"
    )
    print(f"schema ready: {CATALOG}.{schema}")

In [0]:
existing = {r.databaseName for r in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()}
missing = [s for s in SCHEMAS if s not in existing]

if missing:
    raise RuntimeError(f"setup incomplete — missing schemas: {missing}")

print(f"all {len(SCHEMAS)} schemas present in {CATALOG}")
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

In [0]:
# PIPELINE_PRINCIPAL = "<service-principal-or-group>"
# CONSUMER_GROUP = "<analyst-group>"
#
# spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{PIPELINE_PRINCIPAL}`")
# for schema in SCHEMAS:
#     spark.sql(
#         f"GRANT USE SCHEMA, CREATE TABLE, MODIFY, SELECT "
#         f"ON SCHEMA {CATALOG}.{schema} TO `{PIPELINE_PRINCIPAL}`"
#     )
#
# spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{CONSUMER_GROUP}`")
# spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {CATALOG}.gold TO `{CONSUMER_GROUP}`")